In [ ]:

import subprocess
import sys

packages = [
    "pandas",       # DataFrames — think spreadsheets inside Python
    "numpy",        # Fast math on arrays and matrices of numbers
    "yfinance",     # Downloads free stock data from Yahoo Finance
    "ta",           # Technical analysis indicators (RSI, MACD, etc.)
    "matplotlib",   # Creates charts, graphs, and visualizations
    "scikit-learn", # Machine learning (Random Forest, train/test split)
    "requests",     # Makes HTTP requests (downloads web pages properly)
    "lxml",         # HTML parser — required by pd.read_html() to parse tables
]


for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed!")

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import ta
import matplotlib.pyplot as plt
import requests
import base64
from io import BytesIO, StringIO
from datetime import datetime, timedelta
import time
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"📅 Today: {datetime.now().strftime('%A, %B %d, %Y')}")

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/536.36 "
                "(KHTML, like Gecko) CHrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print(f"HTTP Status: {response.status_code}")

if response.status_code == 200:
    tables = pd.read_html(StringIO(response.text))
    sp500_table = tables[0]

    sp500_tickers = [
        str(t).replace('.', '-')
        for t in sp500_table['Symbol'].tolist()
    ]

    sp500_info = sp500_table[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]].copy()
    sp500_info['Symbol'] = sp500_info['Symbol'].str.replace('.', '-', regex=False)
    sp500_info = sp500_info.set_index('Symbol')

    print(f"\n✅ Fetched {len(sp500_tickers)} S&P500 Companies")
    print(f" First 5: {sp500_tickers[:5]}")
    print (f" Last 5: {sp500_tickers[-5:]}")

    print(f"\n Sectors:")
    for sector, count in sp500_info['GICS Sector'].value_counts().items():
        print(f"     {sector:<35} {count} companies")
else:
    print(f"❌ Failed. Status: {response.status_code}. Try again.")

In [ ]:
url = "https://en.wikipedia.org/wiki/Nasdaq-100"
response = requests.get(url,headers=headers)
print(f"HTTP Status: {response.status_code}")

nasdaq_tickers = []

if response.status_code == 200:
    tables = pd.read_html(StringIO(response.text))
    for table in tables:
        cols_lower = [str(c).lower() for c in table.columns]
        if any(word in cols_lower for word in ['ticker', 'symbol']):
            ticker_col = next((c for c in table.columns if str(c).lower() in ('ticker','symbol')), None)
            if ticker_col:
                nasdaq_tickers = [str(t).replace('.', '-') for t in table[ticker_col].tolist()]
                nasdaq_tickers = [t for t in nasdaq_tickers if t.isalpha() or '-' in t]
                break
    if nasdaq_tickers:
        print(f"\n✅ Fetched {len(nasdaq_tickers)} NASDAQ-100 companies")
    else:
        print("Couldn't find NASDSQ-100 table - using S&P500 tech stocks")
        tech_sectors = ["Information Technology", "Communication Services"]
        nasdaq_tickers = sp500_info[sp500_info['GICS Sector'].isin(tech_sectors)].index.tolist()

sp500_set = set(sp500_tickers)
nasdaq_set = set(nasdaq_tickers)
full_universe = sorted(sp500_set | nasdaq_set)

print(f"\n 📈S&P 500: {len(sp500_set)} | NASDAQ-100: {len(nasdaq_set)} | Total unique: {len(full_universe)}")


In [ ]:
market_data = yf.download(["SPY", "^VIX", "QQQ"], period="1y", interval="1d",
                           progress=False, group_by='ticker')

spy_df = market_data["SPY"].dropna().sort_index()
vix_df = market_data["^VIX"].dropna().sort_index()
qqq_df = market_data["QQQ"].dropna().sort_index()

spy_close = spy_df['Close'].squeeze()
vix_close = vix_df['Close'].squeeze()
qqq_close = qqq_df['Close'].squeeze()

spy_price = spy_close.iloc[-1].item()
vix_price = vix_close.iloc[-1].item()
qqq_price = qqq_close.iloc[-1].item()

spy_ret_1d = ((spy_close.iloc[-1] / spy_close.iloc[-2]) - 1).item() * 100
spy_ret_5d = ((spy_close.iloc[-1] / spy_close.iloc[-6]) - 1).item() * 100
spy_ret_20d = ((spy_close.iloc[-1] / spy_close.iloc[-21]) - 1).item() * 100

print("=" * 50)
print("       LIVE MARKET SNAPSHOT")
print("=" * 50)
print(f"  SPY:  ${spy_price:>10,.2f}  |  VIX:  {vix_price:.2f}  |  QQQ:  ${qqq_price:,.2f}")
print(f"  SPY 1d: {spy_ret_1d:+.2f}%  |  5d: {spy_ret_5d:+.2f}%  |  20d: {spy_ret_20d:+.2f}%")
print(f"  Range: {spy_df.index[0].strftime('%Y-%m-%d')} → {spy_df.index[-1].strftime('%Y-%m-%d')}")
print("=" * 50)

In [ ]:
universe = full_universe  # Every unique ticker from S&P 500 + NASDAQ-100

print(f"🎯 Working universe: {len(universe)} stocks")
print(f"   From S&P 500:   {len(sp500_set)} tickers")
print(f"   From NASDAQ-100: {len(nasdaq_set)} tickers")
print(f"   Overlap removed: {len(sp500_set) + len(nasdaq_set) - len(universe)} duplicates")
print(f"\n   First 10: {universe[:10]}")
print(f"   Last 10:  {universe[-10:]}")

In [ ]:
print(f"📥 Downloading data for {len(universe)} stocks (1-3 min)...\n")

all_data = yf.download(universe, period="6mo", interval="1d",
                        progress=False, group_by='ticker')
all_data_1y = yf.download(universe, period="1y", interval="1d",
                           progress=False, group_by='ticker')

print(f"✅ 6mo: {all_data.shape} | 1yr: {all_data_1y.shape}")
sample = universe[0]
print(f"   Sample ({sample}): {len(all_data[sample].dropna())} days, close=${all_data[sample]['Close'].dropna().squeeze().iloc[-1].item():.2f}")

In [ ]:
def analyze_technicals(ticker, data_source):
    """
    Run every technical indicator on one stock using pre-downloaded data.
    Returns a dict with all values, or None if data is insufficient.
    """
    try:
        df = data_source[ticker].dropna().sort_index()
    except KeyError:
        return None
    
    if len(df) < 80:
        return None
    
    close = df['Close'].squeeze()
    volume = df['Volume'].squeeze()
    
    # ── Moving Averages (continuous distance, not binary) ──
    sma_20 = close.rolling(20).mean()
    sma_50 = close.rolling(50).mean()
    latest = close.iloc[-1]
    
    sma_20_dist = float(((latest - sma_20.iloc[-1]) / sma_20.iloc[-1]) * 100) if not pd.isna(sma_20.iloc[-1]) else 0.0
    sma_50_dist = float(((latest - sma_50.iloc[-1]) / sma_50.iloc[-1]) * 100) if not pd.isna(sma_50.iloc[-1]) else 0.0
    above_50 = bool(latest > sma_50.iloc[-1]) if not pd.isna(sma_50.iloc[-1]) else False
    
    # ── RSI ──
    rsi = ta.momentum.RSIIndicator(close, window=14).rsi().iloc[-1]
    
    # ── MACD Histogram (continuous, not binary) ──
    macd_obj = ta.trend.MACD(close)
    macd_hist = float(macd_obj.macd_diff().iloc[-1])
    
    # ── Volume Ratio ──
    vol_avg = volume.rolling(20).mean().iloc[-1]
    vol_ratio = float(volume.iloc[-1] / vol_avg) if vol_avg > 0 else 1.0
    
    # ── Relative Volume Trend (is volume growing?) ──
    vol_5d = volume.rolling(5).mean().iloc[-1]
    vol_20d = volume.rolling(20).mean().iloc[-1]
    rel_vol_trend = float(vol_5d / vol_20d) if vol_20d > 0 else 1.0
    
    # ── Momentum (returns across 3 timeframes) ──
    ret_5d = float(((close.iloc[-1] / close.iloc[-6]) - 1) * 100) if len(close) > 6 else 0.0
    ret_20d = float(((close.iloc[-1] / close.iloc[-21]) - 1) * 100) if len(close) > 21 else 0.0
    ret_60d = float(((close.iloc[-1] / close.iloc[-61]) - 1) * 100) if len(close) > 61 else 0.0
    
    # ── Volatility (smoothness of price movement) ──
    volatility = float(close.pct_change().rolling(20).std().iloc[-1] * 100)
    
    # ── ADX (trend strength, 0-100) ──
    high = df['High'].squeeze()
    low = df['Low'].squeeze()
    adx_obj = ta.trend.ADXIndicator(high, low, close, window=14)
    adx_value = float(adx_obj.adx().iloc[-1]) if not pd.isna(adx_obj.adx().iloc[-1]) else 20.0
    
    # ── ATR % (average daily range as % of price) ──
    atr_obj = ta.volatility.AverageTrueRange(high, low, close, window=14)
    atr_raw = float(atr_obj.average_true_range().iloc[-1])
    atr_pct = (atr_raw / float(latest)) * 100 if latest > 0 else 0.0
    
    # ── OBV Slope (volume conviction trend over 20 days) ──
    obv = ta.volume.OnBalanceVolumeIndicator(close, volume).on_balance_volume()
    obv_slope = float((obv.iloc[-1] - obv.iloc[-21]) / abs(obv.iloc[-21]) * 100) if len(obv) > 21 and abs(obv.iloc[-21]) > 0 else 0.0
    
    return {
        'ticker': ticker, 'price': round(float(latest), 2),
        'sma_20': round(float(sma_20.iloc[-1]), 2),
        'sma_50': round(float(sma_50.iloc[-1]), 2),
        'sma_20_distance': round(sma_20_dist, 2),
        'sma_50_distance': round(sma_50_dist, 2),
        'above_sma_50': above_50,
        'rsi': round(float(rsi), 1),
        'macd_histogram': round(macd_hist, 4),
        'volume_ratio': round(vol_ratio, 2),
        'relative_vol_trend': round(rel_vol_trend, 2),
        'return_5d': round(ret_5d, 2), 'return_20d': round(ret_20d, 2),
        'return_60d': round(ret_60d, 2),
        'volatility': round(volatility, 3),
        'adx': round(adx_value, 1),
        'atr_pct': round(atr_pct, 3),
        'obv_slope': round(obv_slope, 2),
        'momentum_strong': ret_5d > 0 and ret_20d > 0 and ret_60d > 0,
    }

print(f"⚡ Analyzing {len(universe)} stocks...\n")

tech_results = []
for i, ticker in enumerate(universe, 1):
    result = analyze_technicals(ticker, all_data)
    if result:
        tech_results.append(result)
    if i % 50 == 0:
        print(f"   {i}/{len(universe)} analyzed...")

tech_df = pd.DataFrame(tech_results).set_index('ticker')

print(f"\n✅ {len(tech_df)} stocks analyzed")
print(f"\n=== MARKET-WIDE TECHNICAL SUMMARY ===")
print(f"  Above 50 SMA:    {tech_df['above_sma_50'].sum()}/{len(tech_df)} ({tech_df['above_sma_50'].mean():.0%})")
print(f"  Avg SMA20 dist:  {tech_df['sma_20_distance'].mean():+.2f}%")
print(f"  Avg SMA50 dist:  {tech_df['sma_50_distance'].mean():+.2f}%")
print(f"  Avg RSI:         {tech_df['rsi'].mean():.1f}")
print(f"  Avg ADX:         {tech_df['adx'].mean():.1f}")
print(f"  Avg volatility:  {tech_df['volatility'].mean():.2f}%")
print(f"  Avg ATR %:       {tech_df['atr_pct'].mean():.2f}%")
print(f"  Avg 20d return:  {tech_df['return_20d'].mean():+.2f}%")
print(f"  Avg 60d return:  {tech_df['return_60d'].mean():+.2f}%")

In [ ]:
all_features = []
all_labels = []

print(f"📊 Building ML training data from {len(universe)} stocks × 1 year of history...")
print(f"   (This takes 3-5 minutes for the full universe)\n")

for idx, ticker in enumerate(universe, 1):
    try:
        df = all_data_1y[ticker].dropna().sort_index()
        if len(df) < 100:
            continue
        
        close = df['Close'].squeeze()
        volume = df['Volume'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        
        sma_20 = close.rolling(20).mean()
        sma_50 = close.rolling(50).mean()
        rsi_series = ta.momentum.RSIIndicator(close, window=14).rsi()
        macd_hist = ta.trend.MACD(close).macd_diff()
        vol_avg = volume.rolling(20).mean()
        vol_5d_avg = volume.rolling(5).mean()
        daily_returns = close.pct_change()
        volatility_series = daily_returns.rolling(20).std() * 100
        adx_series = ta.trend.ADXIndicator(high, low, close, window=14).adx()
        atr_series = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range()
        obv_series = ta.volume.OnBalanceVolumeIndicator(close, volume).on_balance_volume()
        
        forward_20d = close.pct_change(20).shift(-20)
        
        for i in range(60, len(df) - 20):
            if pd.isna(sma_50.iloc[i]) or pd.isna(rsi_series.iloc[i]):
                continue
            if pd.isna(forward_20d.iloc[i]) or pd.isna(volatility_series.iloc[i]):
                continue
            if pd.isna(adx_series.iloc[i]) or pd.isna(atr_series.iloc[i]):
                continue
            
            features = {
                'sma_20_distance': float(((close.iloc[i] - sma_20.iloc[i]) / sma_20.iloc[i]) * 100),
                'sma_50_distance': float(((close.iloc[i] - sma_50.iloc[i]) / sma_50.iloc[i]) * 100),
                'rsi': float(rsi_series.iloc[i]),
                'macd_histogram': float(macd_hist.iloc[i]),
                'volume_ratio': float(volume.iloc[i] / vol_avg.iloc[i]) if vol_avg.iloc[i] > 0 else 1.0,
                'relative_vol_trend': float(vol_5d_avg.iloc[i] / vol_avg.iloc[i]) if vol_avg.iloc[i] > 0 else 1.0,
                'return_5d': float(((close.iloc[i] / close.iloc[i-5]) - 1) * 100) if i >= 5 else 0,
                'return_20d': float(((close.iloc[i] / close.iloc[i-20]) - 1) * 100) if i >= 20 else 0,
                'return_60d': float(((close.iloc[i] / close.iloc[i-60]) - 1) * 100) if i >= 60 else 0,
                'volatility': float(volatility_series.iloc[i]),
                'adx': float(adx_series.iloc[i]),
                'atr_pct': float(atr_series.iloc[i] / close.iloc[i] * 100),
                'obv_slope': float((obv_series.iloc[i] - obv_series.iloc[i-20]) / abs(obv_series.iloc[i-20]) * 100) if i >= 20 and abs(obv_series.iloc[i-20]) > 0 else 0,
            }
            all_features.append(features)
            all_labels.append(1 if float(forward_20d.iloc[i]) > 0.05 else 0)
    except Exception:
        continue
    
    if idx % 50 == 0:
        print(f"   Processed {idx}/{len(universe)} ({len(all_features)} samples)")

features_df = pd.DataFrame(all_features)
labels = np.array(all_labels)

print(f"\n✅ Built {len(features_df)} training samples")
print(f"   Winners (>5% in 20d): {sum(labels)} ({sum(labels)/len(labels):.1%})")
print(f"   Losers:  {len(labels) - sum(labels)} ({1 - sum(labels)/len(labels):.1%})")
print(f"   Features: {list(features_df.columns)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features_df, labels, test_size=0.2, random_state=42
)

print(f"Training: {len(X_train)} samples | Testing: {len(X_test)} samples")

model = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)

print("\n🧠 Training Random Forest...")
model.fit(X_train, y_train)
print("✅ Training complete!")

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n=== MODEL PERFORMANCE (unseen test data) ===")
print(f"   Accuracy: {accuracy:.1%}\n")
print(classification_report(y_test, y_pred, target_names=['Loser (0)', 'Winner (1)']))

importances = pd.Series(model.feature_importances_, index=features_df.columns
                        ).sort_values(ascending=False)

print("=== FEATURE IMPORTANCE ===\n")
for feat, imp in importances.items():
    bar = "█" * int(imp * 50)
    print(f"   {feat:<22} {imp:.3f}  {bar}")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

winner_probs = model.predict_proba(X_test)[:, 1]
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
rows = []

for t in thresholds:
    y_pred_t = (winner_probs >= t).astype(int)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred_t, average=None, labels=[0, 1], zero_division=0
    )

    predicted_winners = int((y_pred_t == 1).sum())

    rows.append({
        "threshold": t,
        "predicted_winners": predicted_winners,
        "winner_precision": precision[1],
        "winner_recall": recall[1],
        "winner_f1": f1[1]
    })


threshold_df = pd.DataFrame(rows)

display_df = threshold_df.copy()
display_df["winner_precision"] = display_df["winner_precision"].round(3)
display_df["winner_recall"] = display_df["winner_recall"].round(3)
display_df["winner_f1"] = display_df["winner_f1"].round(3)

print("=== THRESHOLD TESTING (Winner class only) ===")
print(display_df.to_string(index=False))

In [ ]:
today_features = []
today_tickers = []

for r in tech_results:
    today_features.append({
        'sma_20_distance': r['sma_20_distance'],
        'sma_50_distance': r['sma_50_distance'],
        'rsi': r['rsi'],
        'macd_histogram': r['macd_histogram'],
        'volume_ratio': r['volume_ratio'],
        'relative_vol_trend': r['relative_vol_trend'],
        'return_5d': r['return_5d'],
        'return_20d': r['return_20d'],
        'return_60d': r['return_60d'],
        'volatility': r['volatility'],
        'adx': r['adx'],
        'atr_pct': r['atr_pct'],
        'obv_slope': r['obv_slope'],
    })
    today_tickers.append(r['ticker'])

today_df = pd.DataFrame(today_features)
win_probs = model.predict_proba(today_df)[:,1]
predictions = (win_probs >= 0.60).astype(int)

ml_df = pd.DataFrame({
    'ticker': today_tickers,
    'ml_prediction': predictions,
    'ml_win_prob': np.round(win_probs, 3)
}).set_index('ticker').sort_values('ml_win_prob', ascending=False)

print(f"=== AI PREDICTIONS (20-day position trades, ≥60% threshold) ===\n")
print(f"   Predicted winners (>5% in 20d): {(ml_df['ml_prediction'] == 1).sum()} / {len(ml_df)}\n")
print(f"   {'Ticker':<7} {'AI Prob':>8} {'Prediction':>12}")
print("   " + "─" * 30)
for t, row in ml_df.head(15).iterrows():
    pred = "🟢 WIN" if row['ml_prediction'] == 1 else "🔴 LOSE"
    print(f"   {t:<7} {row['ml_win_prob']:>7.1%} {pred:>12}")

In [ ]:
def check_earnings_safety (ticker, days_buffer = 21):
    try:
        cal = yf.Ticker(ticker).calendar
        if cal is not None and len(cal) > 0:
            earnings_date = None
            if isinstance(cal, dict):
                ed = cal.get('Earnings Date', None)
                if isinstance(ed, list) and len(ed) > 0:
                    earnings_date = pd.Timestamp(ed[0])
                elif ed is not None:
                    earnings_date = pd.Timestamp(ed)
            elif isinstance(cal, pd.DataFrame) and 'Earnings Date' in cal_columns:
                earnings_date = pd.Timestamp(cal['Earnings Date'].iloc[0])
            if earnings_date is not None:
                days_until = (earnings_date - pd.Timestamp(datetime.now().date())).days
                is_safe = days_until > days_buffer or days_until < 0
                status = f"✔ {days_until}d" if is_safe else f"⚠ {days_until}d"
                return {'is_safe': is_safe, 'status': status}
    except Exception:
        pass
    return {'is safe': True, 'status': '✔ Safe'}

def score_stock_hybrid(ticker, tech, ml_row):
    if tech is None:
        return None
    earnings = check_earnings_safety(ticker)
    score = 0
    signals = []

    d20 = tech['sma_20_distance']
    if 1 <= d20 <= 8: score += 8; signals.append(f"✔ Healthy above 20 SMA ({d20:+.1f}%)")
    elif d20 > 8: score += 3; signals.append(f"⚠ Extended above 20 SMA ({d20:+.1f}%)")
    elif d20 > -2: score += 5; signals.append(f"~ Near 20 SMA ({d20:+.1f}%)")
    else: signals.append(f"✘ Below 20 SMA ({d20:+.1f}%)")

    d50 = tech['sma_50_distance']
    if 1 <= d50 <= 12: score += 8; signals.append(f"✔ Healthy above 50 SMA ({d50:+.1f}%)")
    elif d50 > 12: score += 3; signals.append(f"⚠ Extended above 50 SMA ({d50:+.1f}%)")
    elif d50 < -3: score += 5; signals.append(f"~ Near 50 SMA ({d50:+.1f}%)")
    else: signals.append(f"✘ Below 50 SMA ({d50:+.1f}%)")

    rsi = tech['rsi']
    if 40 <= rsi <= 65:   score += 12; signals.append(f"✔ RSI sweet spot ({rsi:.0f})")
    elif 30 <= rsi < 40 or 65 < rsi <= 70: score += 7; signals.append(f"~ RSI ok ({rsi:.0f})")
    else: score += 2; signals.append(f"⚠ RSI extreme ({rsi:.0f})")

    mh = tech['macd_histogram']
    if mh > 0.5: score += 8; signals.append(f"✔ Strong MACD ({mh:+.2f})")
    elif mh > 0: score += 5; signals.append(f"✔ MACD positive ({mh:+.2f})")
    else: signals.append(f"✘ MACD negative ({mh:+.2f})")
    
    vol = tech['volume_ratio']
    vt = tech['relative_vol_trend']
    if vol > 1.3 and vt > 1.1: score += 8; signals.append(f"✔ Strong rising volume ({vol:.1f}x, trend {vt:.2f})")
    elif vol > 1.0: score += 5; signals.append(f"✔ Above avg volume ({vol:.1f}x)")
    else: score += 1; signals.append(f"✘ Low volume ({vol:.1f}x)")
    
    r5 = tech['return_5d']
    r20 = tech['return_20d']
    r60 = tech['return_60d']
    if r5 > 0 and r20 > 3 and r60 > 5:
        score += 12; signals.append(f"✔ Strong trend all timeframes (+{r60:.1f}% 60d)")
    elif r20 > 0 and r60 > 0:
        score += 8; signals.append(f"✔ Positive medium/long momentum")
    elif r60 > 0:
        score += 4; signals.append(f"~ Long-term positive only (+{r60:.1f}% 60d)")
    else: signals.append(f"✘ Negative momentum ({r60:+.1f}% 60d)")
    
    # ── Volatility (lower is better for position trades) ──
    v = tech['volatility']
    if v < 1.5: score += 6; signals.append(f"✔ Low volatility ({v:.2f}%)")
    elif v < 2.5: score += 3; signals.append(f"✔ Moderate volatility ({v:.2f}%)")
    else: score += 1; signals.append(f"⚠ High volatility ({v:.2f}%)")
    
    # ── ADX Trend Strength (0-8 pts) ──
    adx = tech['adx']
    if adx >= 30:   score += 8; signals.append(f"✔ Strong trend (ADX {adx:.0f})")
    elif adx >= 20: score += 5; signals.append(f"~ Developing trend (ADX {adx:.0f})")
    else:           score += 1; signals.append(f"✘ No trend (ADX {adx:.0f})")
    
    # ── OBV Volume Conviction (0-4 pts) ──
    obv = tech['obv_slope']
    if obv > 5:   score += 4; signals.append(f"✔ Volume accumulation (OBV +{obv:.0f}%)")
    elif obv > 0: score += 2; signals.append(f"~ OBV flat ({obv:+.0f}%)")
    else: signals.append(f"✘ Volume distribution (OBV {obv:+.0f}%)")
    
    # ── ATR-Based Stop Loss (smarter than fixed %) ──
    atr = tech['atr_pct']
    if atr < 2.0: signals.append(f"✔ Tight range (ATR {atr:.1f}%)")
    elif atr < 4.0: signals.append(f"~ Normal range (ATR {atr:.1f}%)")
    else: signals.append(f"⚠ Wide range (ATR {atr:.1f}%)")
    
    # ── Earnings Safety ──
    if earnings['is_safe']: score += 8; signals.append(f"✔ Earnings: {earnings['status']}")
    else: signals.append(f"⚠ Earnings: {earnings['status']}")

    ml_prob = ml_row.get('ml_win_prob', 0.5)
    ml_pts = round(ml_prob * 30)
    score += ml_pts
    signals.append(f"🤖 AI confidence: {ml_prob:.0%} (+{ml_pts}pts)")

    price = tech['price']
    atr_dollar = price * (tech['atr_pct'] / 100)
    stop = round(price - atr_dollar * 3, 2)
    target = round(price + (price - stop) * 2.5, 2)
    risk = price - stop
    reward = target - price
    rr = f"1: {round(reward / risk, 1)}" if risk > 0 else "N/A"
    conf = "HIGH" if score >= 80 else "MEDIUM" if score >= 65 else "LOW"

    return {
        'ticker': ticker, 'score': score, 'ml_prob': ml_prob,
        'signals': signals, 'price': price, 'entry': price,
        'stop_loss': stop, 'target': target, 'risk_reward': rr,
        'earnings_safe': earnings['is_safe'], 'earnings_status': earnings['status'],
        'confidence': conf, 'technicals': tech,
    }
    
print(f"⚡ Scoring {len(tech_results)} stocks (checking earnings dates — 3-5 min)...\n")

all_scores = []

for j, tech in enumerate(tech_results, 1):
    t = tech['ticker']
    ml_row = ml_df.loc[t].to_dict() if t in ml_df.index else {'ml_win_prob': 0.5}
    result = score_stock_hybrid(t, tech, ml_row)
    if result:
        all_scores.append(result)
    if j % 50 == 0:
        print(f"   {j}/{len(tech_results)} scored...")
    time.sleep(0.05)
    all_scores.sort(key=lambda x:x['score'], reverse=True)
    qualified = [s for s in all_scores if s['score'] >= 65 and s['earnings_safe']]
    print(f"✅ {len(all_scores)} scored | {len(qualified)} qualified\n")
    medals = {1: "🥇", 2: "🥈", 3: "🥉"}
    for i, s in enumerate(all_scores[:15], 1):
        m = medals.get(i, f"  {i}")
        print(f"  {m:<5} {s['ticker']:<7} Score: {s['score']:>3}  AI: {s['ml_prob']:.0%}  {s['confidence']}")
        

In [ ]:
def detect_market_regime (spy_close, vix_close, tech_df):
    latest_spy = spy_close.iloc[-1]

    s20 = bool(latest_spy > spy_close.rolling(20).mean().iloc[-1])
    s50 = bool(latest_spy > spy_close.rolling(50).mean().iloc[-1])
    s200_val = spy_close.rolling(200).mean().iloc[-1]
    s200 = bool(latest_spy > s200_val) if not pd.isna(s200_val) else False
    spy_score = int(s20) + int(s50) + int(s200)
    spy_sig = "🟢 Bullish" if spy_score == 3 else "🟡 Mixed" if spy_score >= 2 else "🔴 Bearish"

    vix_val = float(vix_close.iloc[-1])
    if vix_val < 15: vix_score, vix_sig = 3, "🟢 Calm"
    elif vix_val < 20: vix_score, vix_sig = 2, "🟡 Elevated"
    elif vix_val < 30: vix_score, vix_sig = 1, "🟠 High"
    else: vix_score, vix_sig = 0, "🔴 Extreme"

    breadth_pct = tech_df['above_sma_50'].mean()
    if breadth_pct > 0.7: b_score, b_sig = 2, "🟢 Strong"
    elif breadth_pct > 0.5: b_score, b_sig = 1, "🟡 Neutral"
    else: b_score, b_sig = 0, "🔴 Weak"

    r5 = float(((spy_close.iloc[-1] / spy_close.iloc[-6]) - 1) * 100)
    r20 = float(((spy_close.iloc[-1] / spy_close.iloc[-21]) - 1) * 100)
    if r5 > 0 and r20 > 0: m_score, m_sig = 2, "🟢 Strong"
    elif r5 > 0 or r20 > 0: m_score, m_sig = 1, "🟡 Mixed"
    else: m_score, m_sig = 0, "🔴 Weak"

    total = spy_score + vix_score + b_score + m_score
    if total >= 8: regime_label = "🟢 RISK-ON"
    elif total >= 5: regime_label = "🟡 NEUTRAL"
    else: regime_label = "🔴 RISK-OFF"

    return {
        'regime': regime_label, 'score': total,
        'spy_price': round(float(latest_spy), 2), 'spy_trend': spy_sig, 'spy_trend_score': spy_score,
        'vix': round(vix_val, 2), 'vix_signal': vix_sig, 'vix_score': vix_score,
        'breadth_pct': round(breadth_pct * 100, 1), 'breadth_signal': b_sig, 'breadth_score': b_score,
        'spy_5d_return': round(r5, 2), 'momentum_signal': m_sig, 'momentum_score': m_score,
        }
def calculate_position_size(account, risk_pct, entry, stop):
    """Calculate shares to buy so you risk exactly risk_pct of account."""
    dollar_risk = account * risk_pct
    risk_per_share = entry - stop
    if risk_per_share <= 0: return None
    shares = int(dollar_risk / risk_per_share)
    return {'shares': shares, 'value': round(shares * entry, 2),
            'pct': round(shares * entry / account * 100, 1), 'risk': round(dollar_risk, 2)}

regime = detect_market_regime(spy_close, vix_close, tech_df)

if regime['score'] >= 8: adj_risk, mode = 0.02, "FULL"
elif regime['score'] >= 5: adj_risk, mode = 0.01, "HALF"
else: adj_risk, mode = 0, "NONE"

print(f"╔{'═' * 50}╗")
print(f"║{'MARKET REGIME':^50}║")
print(f"║  {regime['regime']:<20} Score: {regime['score']}/10{' ' * 14}║")
print(f"║  SPY: {regime['spy_trend']:<15} VIX: {regime['vix']} {regime['vix_signal']:<10}║")
print(f"║  Breadth: {regime['breadth_signal']:<11} Momentum: {regime['momentum_signal']:<12}║")
print(f"║  Risk Mode: {mode:<10} Capital/Trade: {adj_risk:.1%}{' ' * 10}║")
print(f"╚{'═' * 50}╝")
        
        


In [ ]:
def chart_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                facecolor='#0f172a', edgecolor='none')
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return img_str


close_c = spy_close[-252:]
vol_c = spy_df['Volume'].squeeze()[-252:]

fig1, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
    gridspec_kw={'height_ratios': [3, 1]}, facecolor='#0f172a')

ax1.plot(close_c.index, close_c, color='#38bdf8', linewidth=1.8, label='Price')
ax1.plot(close_c.index, close_c.rolling(20).mean(), color='#facc15', linewidth=1, alpha=0.7, label='20 SMA')
ax1.plot(close_c.index, close_c.rolling(50).mean(), color='#f97316', linewidth=1, alpha=0.7, label='50 SMA')
ax1.fill_between(close_c.index, close_c, close_c.rolling(20).mean(),
    where=(close_c > close_c.rolling(20).mean()), alpha=0.08, color='#22c55e')
ax1.fill_between(close_c.index, close_c, close_c.rolling(20).mean(),
    where=(close_c < close_c.rolling(20).mean()), alpha=0.08, color='#ef4444')
for s in ['top','right']: ax1.spines[s].set_visible(False)
ax1.set_facecolor('#0f172a'); ax1.tick_params(colors='#94a3b8')
ax1.legend(loc='upper left', fontsize=8, facecolor='#1e293b', edgecolor='none', labelcolor='#e2e8f0')
ax1.set_title('SPY — Price & Moving Averages', color='#e2e8f0', fontsize=14, pad=12)

vc = ['#22c55e' if close_c.iloc[i] >= close_c.iloc[max(0,i-1)] else '#ef4444' for i in range(len(close_c))]
ax2.bar(close_c.index, vol_c, color=vc, alpha=0.6, width=1)
for s in ['top','right']: ax2.spines[s].set_visible(False)
ax2.set_facecolor('#0f172a'); ax2.tick_params(colors='#94a3b8')
ax2.set_title('Volume', color='#94a3b8', fontsize=10)

plt.tight_layout()
chart1 = chart_to_base64(fig1)


top20 = all_scores[:20]
tk_list = [s['ticker'] for s in top20][::-1]
sc_list = [s['score'] for s in top20][::-1]
colors2 = ['#22c55e' if s >= 80 else '#facc15' if s >= 65 else '#ef4444' for s in sc_list]

fig2, ax3 = plt.subplots(figsize=(10, 7), facecolor='#0f172a')
ax3.barh(tk_list, sc_list, color=colors2, edgecolor='none', height=0.7)
ax3.set_xlim(0, 100)
for s in ['top','right']: ax3.spines[s].set_visible(False)
ax3.set_facecolor('#0f172a'); ax3.tick_params(colors='#94a3b8')
ax3.set_title('Top 20 — Hybrid Score Rankings', color='#e2e8f0', fontsize=14, pad=12)
for i, v in enumerate(sc_list):
    ax3.text(v + 1, i, str(v), color='#e2e8f0', va='center', fontsize=9)

plt.tight_layout()
chart2 = chart_to_base64(fig2)


high_c = len([s for s in all_scores if s['score'] >= 80])
med_c = len([s for s in all_scores if 65 <= s['score'] < 80])
low_c = len([s for s in all_scores if s['score'] < 65])

fig3, ax4 = plt.subplots(figsize=(7, 7), facecolor='#0f172a')
wedges, texts, autotexts = ax4.pie(
    [high_c, med_c, low_c], labels=['HIGH', 'MEDIUM', 'LOW'],
    colors=['#22c55e', '#facc15', '#ef4444'], autopct='%1.0f%%',
    textprops={'color': '#e2e8f0', 'fontsize': 12})
ax4.set_title('Score Distribution', color='#e2e8f0', fontsize=14, pad=12)

plt.tight_layout()
chart3 = chart_to_base64(fig3)

feat_names = features_df.columns.tolist()
feat_imps = model.feature_importances_
sorted_idx = np.argsort(feat_imps)

fig4, ax5 = plt.subplots(figsize=(10, 6), facecolor='#0f172a')
ax5.barh([feat_names[i] for i in sorted_idx], feat_imps[sorted_idx],
         color='#38bdf8', edgecolor='none', height=0.6)
for s in ['top','right']: ax5.spines[s].set_visible(False)
ax5.set_facecolor('#0f172a'); ax5.tick_params(colors='#94a3b8')
ax5.set_title('ML Feature Importance', color='#e2e8f0', fontsize=14, pad=12)

plt.tight_layout()
chart4 = chart_to_base64(fig4)

chart_images = {'price_sma': chart1, 'rankings': chart2,
                'distribution': chart3, 'features': chart4}
print(f"✅ {len(chart_images)} charts generated and encoded")

In [ ]:
ACCOUNT_SIZE = 25000  # ← Change this to your actual account balance

today_str = datetime.now().strftime("%B %d, %Y")
today_short = datetime.now().strftime("%Y-%m-%d %H:%M")
top_picks = qualified[:5]
report_path = "position_ai_report.html"


top_feat_idx = model.feature_importances_.argmax()
top_feat_name = features_df.columns[top_feat_idx]
n_train = len(features_df)
max_positions = min(8, len(qualified))


if regime['score'] >= 8:
    r_grad = "linear-gradient(135deg, #065f46 0%, #064e3b 50%, #022c22 100%)"
    r_badge_bg = "#059669"; r_badge_label = "RISK-ON"
    regime_bar_color = "linear-gradient(90deg,#059669,#34d399)"
elif regime['score'] >= 5:
    r_grad = "linear-gradient(135deg, #92400e 0%, #78350f 50%, #451a03 100%)"
    r_badge_bg = "#d97706"; r_badge_label = "NEUTRAL"
    regime_bar_color = "linear-gradient(90deg,#d97706,#fbbf24)"
else:
    r_grad = "linear-gradient(135deg, #991b1b 0%, #7f1d1d 50%, #450a0a 100%)"
    r_badge_bg = "#dc2626"; r_badge_label = "RISK-OFF"
    regime_bar_color = "linear-gradient(90deg,#dc2626,#f87171)"


def score_color(sc):
    if sc >= 80: return "#22c55e"
    elif sc >= 65: return "#fbbf24"
    else: return "#ef4444"

def conf_badge(conf):
    if conf == "HIGH":
        return '<span style="background:#166534;color:#bbf7d0;padding:3px 10px;border-radius:8px;font-size:0.8em;font-weight:600">HIGH</span>'
    elif conf == "MEDIUM":
        return '<span style="background:#854d0e;color:#fef08a;padding:3px 10px;border-radius:8px;font-size:0.8em;font-weight:600">MEDIUM</span>'
    else:
        return '<span style="background:#7f1d1d;color:#fca5a5;padding:3px 10px;border-radius:8px;font-size:0.8em;font-weight:600">LOW</span>'

def bar_color(pct):
    if pct >= 80: return "#22c55e"
    elif pct >= 50: return "#fbbf24"
    else: return "#ef4444"


S_MONO = "font-family:'JetBrains Mono',monospace;"
S_TD = f"padding:12px 14px;color:#cbd5e1;border-bottom:1px solid rgba(30,41,59,0.5);{S_MONO}"
S_TD_PLAIN = "padding:12px 14px;color:#cbd5e1;border-bottom:1px solid rgba(30,41,59,0.5);"
S_TH = "text-align:left;padding:12px 14px;font-size:0.72em;font-weight:700;text-transform:uppercase;letter-spacing:1.2px;color:#64748b;border-bottom:2px solid #1e293b;"
S_CARD = "background:#0f172a;border:1px solid #1e293b;border-radius:14px;padding:24px;margin-bottom:16px;"
S_KPI = "background:#0f172a;border:1px solid #1e293b;border-radius:14px;padding:22px 20px;text-align:center;"
S_KPI_VAL = f"font-size:2.1em;font-weight:800;{S_MONO}margin-bottom:4px;"
S_KPI_LABEL = "font-size:0.75em;color:#64748b;text-transform:uppercase;letter-spacing:1.2px;font-weight:600;"
S_SECTION_T = "font-size:0.78em;font-weight:700;text-transform:uppercase;letter-spacing:2px;color:#64748b;margin-bottom:16px;padding-bottom:10px;border-bottom:1px solid #1e293b;"
S_REGIME_ITEM = "background:#0f172a;border:1px solid #1e293b;border-radius:14px;padding:20px;text-align:center;"
S_REGIME_LABEL = "font-size:0.7em;color:#64748b;text-transform:uppercase;letter-spacing:1.2px;margin-bottom:8px;font-weight:600;"
S_REGIME_VAL = f"font-size:1.6em;font-weight:700;{S_MONO}"
S_PROGRESS_TRACK = "width:100%;height:6px;background:#1e293b;border-radius:3px;margin-top:10px;overflow:hidden;"
S_SIGNAL_ITEM = "padding:7px 0;font-size:0.95em;display:flex;align-items:center;gap:8px;"
S_TF = "background:rgba(30,41,59,0.6);border-radius:10px;padding:14px;text-align:center;"
S_TF_LABEL = "font-size:0.68em;color:#64748b;text-transform:uppercase;letter-spacing:1.2px;margin-bottom:4px;"
S_TF_VAL = f"font-size:1.35em;font-weight:700;{S_MONO}color:#f1f5f9;"


picks_rows = ""
medals = {0: "🥇", 1: "🥈", 2: "🥉"}
for i, s in enumerate(top_picks):
    m = medals.get(i, f'<span style="color:#64748b">{i+1}</span>')
    sc = s['score']
    sc_c = score_color(sc)
    picks_rows += f"""
                <tr style="background:#0f172a;">
                    <td style="{S_TD_PLAIN}text-align:center;font-size:1.3em">{m}</td>
                    <td style="{S_TD_PLAIN}"><strong style="color:#f1f5f9;">{s['ticker']}</strong></td>
                    <td style="{S_TD_PLAIN}"><span style="background:{sc_c};color:#0f172a;padding:3px 11px;border-radius:12px;font-weight:700;font-size:0.85em">{sc}</span></td>
                    <td style="{S_TD}color:#cbd5e1;">{s['ml_prob']:.0%}</td>
                    <td style="{S_TD}text-align:right;color:#cbd5e1;">&#36;{s['price']:,.2f}</td>
                    <td style="{S_TD}color:#cbd5e1;">{s['risk_reward']}</td>
                    <td style="{S_TD_PLAIN}color:#cbd5e1;">{s['earnings_status']}</td>
                    <td style="{S_TD_PLAIN}">{conf_badge(s['confidence'])}</td>
                </tr>"""


signals_html = ""
if top_picks:
    s1 = top_picks[0]
    for sig in s1['signals']:
        if sig.startswith("✔") or sig.startswith("🤖"):
            sig_color = "#4ade80" if not sig.startswith("🤖") else "#a78bfa"
        elif sig.startswith("⚠"):
            sig_color = "#fbbf24"
        else:
            sig_color = "#64748b"
        signals_html += f'<div style="{S_SIGNAL_ITEM}color:{sig_color}">{sig}</div>'

    pos1 = calculate_position_size(ACCOUNT_SIZE, adj_risk, s1['entry'], s1['stop_loss'])
    pos1_html = ""
    if pos1 and pos1['shares'] > 0:
        pos1_html = f"""
        <div style="margin-top:18px;padding:18px;background:rgba(56,189,248,0.06);border-left:3px solid #38bdf8;border-radius:0 10px 10px 0;">
            <div style="color:#7dd3fc;font-size:0.78em;text-transform:uppercase;letter-spacing:1.2px;margin-bottom:6px;font-weight:600;">Suggested Position (&#36;{ACCOUNT_SIZE:,} Account)</div>
            <div style="color:#e2e8f0;font-size:1.05em">
                <strong style="color:#e2e8f0;{S_MONO}">{pos1['shares']} shares</strong> = &#36;{pos1['value']:,.0f} ({pos1['pct']}% of portfolio)
                &nbsp;·&nbsp; Max risk: &#36;{pos1['risk']:,.0f}
            </div>
        </div>"""
    elif adj_risk == 0:
        pos1_html = """
        <div style="margin-top:18px;padding:18px;background:rgba(239,68,68,0.06);border-left:3px solid #ef4444;border-radius:0 10px 10px 0;">
            <div style="color:#fca5a5;font-size:0.78em;text-transform:uppercase;letter-spacing:1.2px;margin-bottom:6px;font-weight:600;">Position Sizing</div>
            <div style="color:#f87171;font-size:1.05em;font-weight:600;">
                ⛔ RISK-OFF — No positions recommended in current regime
            </div>
        </div>"""

    signals_html += f"""
        <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:20px;">
            <div style="{S_TF}">
                <div style="{S_TF_LABEL}">Entry</div>
                <div style="{S_TF_VAL}color:#4ade80;">&#36;{s1['entry']:,.2f}</div>
            </div>
            <div style="{S_TF}">
                <div style="{S_TF_LABEL}">Stop Loss</div>
                <div style="{S_TF_VAL}color:#f87171;">&#36;{s1['stop_loss']:,.2f}</div>
            </div>
            <div style="{S_TF}">
                <div style="{S_TF_LABEL}">Target</div>
                <div style="{S_TF_VAL}color:#38bdf8;">&#36;{s1['target']:,.2f}</div>
            </div>
            <div style="{S_TF}">
                <div style="{S_TF_LABEL}">R / R</div>
                <div style="{S_TF_VAL}color:#e2e8f0;">{s1['risk_reward']}</div>
            </div>
        </div>
        {pos1_html}"""


rank_rows = ""
qualified_tickers = set(s['ticker'] for s in qualified)
for idx, s in enumerate(all_scores[:20]):
    is_qual = s['ticker'] in qualified_tickers
    tech = s.get('technicals', {})
    rsi_val = tech.get('rsi', 0)
    vol_ratio = tech.get('volume_ratio', 0)
    mom_5d = tech.get('momentum_5d', 0)

    if s['score'] >= 80:
        row_bg = "background:rgba(34,197,94,0.06);"
    elif s['score'] >= 65:
        row_bg = "background:rgba(251,191,36,0.04);"
    else:
        row_bg = "background:#0f172a;"

    arrow = '<span style="color:#4ade80;font-weight:bold">→</span> ' if is_qual else ''
    mom_color = "#4ade80" if mom_5d >= 0 else "#f87171"

    rank_rows += f"""
                <tr style="{row_bg}">
                    <td style="{S_TD_PLAIN}color:#64748b;">{arrow}{idx+1}</td>
                    <td style="{S_TD_PLAIN}"><strong style="color:#f1f5f9;">{s['ticker']}</strong></td>
                    <td style="{S_TD}text-align:center;color:#cbd5e1;">{s['score']}</td>
                    <td style="{S_TD}text-align:center;color:#cbd5e1;">{s['ml_prob']:.0%}</td>
                    <td style="{S_TD}text-align:right;color:#cbd5e1;">&#36;{s['price']:,.2f}</td>
                    <td style="{S_TD}text-align:right;color:{mom_color};">{mom_5d:+.1f}%</td>
                    <td style="{S_TD}text-align:center;color:#cbd5e1;">{rsi_val:.0f}</td>
                    <td style="{S_TD}text-align:center;color:#cbd5e1;">{vol_ratio:.1f}x</td>
                    <td style="{S_TD_PLAIN}text-align:center;color:#cbd5e1;">{s['earnings_status']}</td>
                </tr>"""


regime_pct = regime['score'] * 10
spy_trend_pct = round((regime.get('spy_trend_score', 0) / 3) * 100)
vix_pct = round((regime.get('vix_score', 0) / 3) * 100)
breadth_pct_bar = round(regime['breadth_pct'])
mom_pct = round((regime.get('momentum_score', 0) / 2) * 100)


spy_val_color = "#4ade80" if spy_trend_pct >= 80 else "#fbbf24" if spy_trend_pct >= 50 else "#f87171"
vix_val_color = "#34d399"
breadth_val_color = "#fbbf24" if breadth_pct_bar < 70 else "#4ade80"
mom_val_color = "#4ade80" if mom_pct >= 80 else "#fbbf24" if mom_pct >= 50 else "#f87171"
risk_mode_color = "#4ade80" if mode == "FULL" else "#fbbf24" if mode == "HALF" else "#f87171"


html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Swing AI Daily Report — {today_str}</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ font-family: 'Plus Jakarta Sans', -apple-system, sans-serif; background: #030712; color: #e2e8f0; line-height: 1.6; -webkit-font-smoothing: antialiased; }}
    .report-header::before {{ content: ''; position: absolute; top: -60%; right: -15%; width: 550px; height: 550px; background: radial-gradient(circle, rgba(52,211,153,0.08) 0%, transparent 70%); border-radius: 50%; }}
    .report-header::after {{ content: ''; position: absolute; bottom: -40%; left: -10%; width: 400px; height: 400px; background: radial-gradient(circle, rgba(16,185,129,0.05) 0%, transparent 70%); border-radius: 50%; }}
    tbody tr:hover {{ background: rgba(56,189,248,0.04) !important; }}
    @media(max-width:768px) {{
        .regime-grid {{ grid-template-columns: repeat(2, 1fr) !important; }}
        .trade-plan-grid {{ grid-template-columns: repeat(2, 1fr) !important; }}
        .chart-grid {{ grid-template-columns: 1fr !important; }}
    }}
</style>
</head>
<body style="font-family:'Plus Jakarta Sans',-apple-system,sans-serif;background:#030712;color:#e2e8f0;line-height:1.6;margin:0;padding:0;">
<div style="max-width:1100px;margin:0 auto;padding:0 24px 80px;">

<!-- ═══════════════════════════════ HEADER ═══════════════════════════════ -->
<div class="report-header" style="background:{r_grad};padding:52px 44px 44px;margin:0 -24px 44px;position:relative;overflow:hidden;">
    <h1 style="font-size:2.4em;font-weight:800;letter-spacing:-0.5px;margin-bottom:4px;position:relative;color:#f8fafc;">Swing AI Daily Report</h1>
    <div style="font-size:1.05em;opacity:0.65;margin-bottom:22px;font-weight:400;position:relative;color:#e2e8f0;">{today_str} &nbsp;·&nbsp; Model v2.0 (Random Forest + Technical Analysis)</div>
    <div style="display:flex;gap:10px;flex-wrap:wrap;position:relative;">
        <span style="padding:5px 15px;border-radius:20px;font-size:0.82em;font-weight:600;background:{r_badge_bg};color:white;box-shadow:0 0 20px rgba(5,150,105,0.3);">{r_badge_label}</span>
        <span style="padding:5px 15px;border-radius:20px;font-size:0.82em;font-weight:600;border:1px solid rgba(255,255,255,0.2);color:rgba(255,255,255,0.8);">Regime {regime['score']}/10</span>
        <span style="padding:5px 15px;border-radius:20px;font-size:0.82em;font-weight:600;border:1px solid rgba(255,255,255,0.2);color:rgba(255,255,255,0.8);">{len(all_scores)} Stocks Scanned</span>
        <span style="padding:5px 15px;border-radius:20px;font-size:0.82em;font-weight:600;border:1px solid rgba(255,255,255,0.2);color:rgba(255,255,255,0.8);">{len(qualified)} Qualified Trades</span>
        <span style="padding:5px 15px;border-radius:20px;font-size:0.82em;font-weight:600;border:1px solid rgba(255,255,255,0.2);color:rgba(255,255,255,0.8);">ML Accuracy: {accuracy:.1%}</span>
    </div>
</div>

<!-- ═══════════════════════════════ KPI CARDS ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Executive Summary</div>
    <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:14px;margin-bottom:16px;">
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#4ade80;">{regime['score']}/10</div>
            <div style="{S_KPI_LABEL}">Regime Score</div>
            <div style="{S_PROGRESS_TRACK}"><div style="height:100%;border-radius:3px;width:{regime_pct}%;background:{regime_bar_color}"></div></div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#38bdf8;">{len(qualified)}</div>
            <div style="{S_KPI_LABEL}">Qualified Trades</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#a78bfa;">{top_picks[0]['ticker'] if top_picks else 'N/A'}</div>
            <div style="{S_KPI_LABEL}">Top Opportunity</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#fbbf24;">{accuracy:.1%}</div>
            <div style="{S_KPI_LABEL}">Model Test Accuracy</div>
        </div>
    </div>
</div>

<!-- ═══════════════════════════════ MARKET REGIME ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Market Environment</div>
    <div class="regime-grid" style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:16px;">
        <div style="{S_REGIME_ITEM}">
            <div style="{S_REGIME_LABEL}">SPY Trend</div>
            <div style="{S_REGIME_VAL}color:{spy_val_color};">{regime.get('spy_trend_score', 0)}/3</div>
            <div style="font-size:0.88em;margin-top:6px;color:#e2e8f0;">{regime['spy_trend']}</div>
            <div style="{S_PROGRESS_TRACK}"><div style="height:100%;border-radius:3px;width:{spy_trend_pct}%;background:{bar_color(spy_trend_pct)}"></div></div>
        </div>
        <div style="{S_REGIME_ITEM}">
            <div style="{S_REGIME_LABEL}">VIX Level</div>
            <div style="{S_REGIME_VAL}color:{vix_val_color};">{regime['vix']}</div>
            <div style="font-size:0.88em;margin-top:6px;color:#e2e8f0;">{regime['vix_signal']}</div>
            <div style="{S_PROGRESS_TRACK}"><div style="height:100%;border-radius:3px;width:{vix_pct}%;background:{bar_color(vix_pct)}"></div></div>
        </div>
        <div style="{S_REGIME_ITEM}">
            <div style="{S_REGIME_LABEL}">Breadth</div>
            <div style="{S_REGIME_VAL}color:{breadth_val_color};">{regime['breadth_pct']}%</div>
            <div style="font-size:0.88em;margin-top:6px;color:#e2e8f0;">{regime['breadth_signal']}</div>
            <div style="{S_PROGRESS_TRACK}"><div style="height:100%;border-radius:3px;width:{breadth_pct_bar}%;background:{bar_color(breadth_pct_bar)}"></div></div>
        </div>
        <div style="{S_REGIME_ITEM}">
            <div style="{S_REGIME_LABEL}">SPY Momentum</div>
            <div style="{S_REGIME_VAL}color:{mom_val_color};">{regime.get('momentum_score', 0)}/2</div>
            <div style="font-size:0.88em;margin-top:6px;color:#e2e8f0;">{regime['momentum_signal']}</div>
            <div style="{S_PROGRESS_TRACK}"><div style="height:100%;border-radius:3px;width:{mom_pct}%;background:{bar_color(mom_pct)}"></div></div>
        </div>
    </div>
</div>

<!-- ═══════════════════════════════ TOP PICKS TABLE ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Top Buy Opportunities</div>
    <div style="{S_CARD}overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:0.9em;">
            <thead>
                <tr style="background:#0f172a;">
                    <th style="{S_TH}text-align:center;">Rank</th>
                    <th style="{S_TH}">Ticker</th>
                    <th style="{S_TH}">Score</th>
                    <th style="{S_TH}">AI Prob</th>
                    <th style="{S_TH}text-align:right;">Price</th>
                    <th style="{S_TH}">R / R</th>
                    <th style="{S_TH}">Earnings</th>
                    <th style="{S_TH}">Confidence</th>
                </tr>
            </thead>
            <tbody>{picks_rows}
            </tbody>
        </table>
    </div>
</div>

<!-- ═══════════════════════════════ #1 SIGNAL BREAKDOWN ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">{top_picks[0]['ticker'] if top_picks else 'N/A'} — Signal Breakdown</div>
    <div style="background:#0f172a;border:1px solid #1e293b;border-radius:14px;padding:28px;">
        {signals_html}
    </div>
</div>

<!-- ═══════════════════════════════ CHARTS ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Market Overview</div>
    <div style="{S_CARD}">
        <img style="width:100%;border-radius:10px;display:block;" src="data:image/png;base64,{chart_images['price_sma']}">
    </div>
</div>

<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Analytics</div>
    <div class="chart-grid" style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
        <div style="{S_CARD}"><img style="width:100%;border-radius:10px;display:block;" src="data:image/png;base64,{chart_images['rankings']}"></div>
        <div style="{S_CARD}"><img style="width:100%;border-radius:10px;display:block;" src="data:image/png;base64,{chart_images['distribution']}"></div>
    </div>
    <div style="{S_CARD}margin-top:16px;">
        <img style="width:100%;border-radius:10px;display:block;" src="data:image/png;base64,{chart_images['features']}">
    </div>
</div>

<!-- ═══════════════════════════════ RISK PANEL ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Risk Control Panel</div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:16px;">
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:{risk_mode_color};font-size:1.5em;">{mode}</div>
            <div style="{S_KPI_LABEL}">Risk Mode</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#e2e8f0;font-size:1.5em;">{adj_risk:.1%}</div>
            <div style="{S_KPI_LABEL}">Capital Per Trade</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#e2e8f0;font-size:1.5em;">{max_positions}</div>
            <div style="{S_KPI_LABEL}">Max Positions</div>
        </div>
    </div>
</div>

<!-- ═══════════════════════════════ MODEL DIAGNOSTICS ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Model Diagnostics</div>
    <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:16px;">
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#38bdf8;font-size:1.2em;">RF-{model.n_estimators}</div>
            <div style="{S_KPI_LABEL}">Model</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#e2e8f0;font-size:1.2em;">{n_train:,}</div>
            <div style="{S_KPI_LABEL}">Training Samples</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#4ade80;font-size:1.2em;">{accuracy:.1%}</div>
            <div style="{S_KPI_LABEL}">Test Accuracy</div>
        </div>
        <div style="{S_KPI}">
            <div style="{S_KPI_VAL}color:#a78bfa;font-size:1.2em;">{top_feat_name}</div>
            <div style="{S_KPI_LABEL}">Top Feature</div>
        </div>
    </div>
</div>

<!-- ═══════════════════════════════ FULL RANKINGS ═══════════════════════════════ -->
<div style="margin-bottom:36px;">
    <div style="{S_SECTION_T}">Complete Rankings — Top 20</div>
    <div style="{S_CARD}overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:0.9em;">
            <thead>
                <tr style="background:#0f172a;">
                    <th style="{S_TH}">#</th>
                    <th style="{S_TH}">Ticker</th>
                    <th style="{S_TH}text-align:center;">Score</th>
                    <th style="{S_TH}text-align:center;">AI Prob</th>
                    <th style="{S_TH}text-align:right;">Price</th>
                    <th style="{S_TH}text-align:right;">5d Mom</th>
                    <th style="{S_TH}text-align:center;">RSI</th>
                    <th style="{S_TH}text-align:center;">Volume</th>
                    <th style="{S_TH}text-align:center;">Earnings</th>
                </tr>
            </thead>
            <tbody>{rank_rows}
            </tbody>
        </table>
    </div>
</div>

<!-- ═══════════════════════════════ FOOTER ═══════════════════════════════ -->
<div style="margin-top:52px;padding:28px 0;border-top:1px solid #1e293b;text-align:center;color:#475569;font-size:0.82em;">
    <div style="margin-bottom:8px;color:#94a3b8;">
        <strong style="color:#94a3b8;">Swing AI Daily Report</strong> &nbsp;·&nbsp; Generated {today_short}
    </div>
    <div style="color:#64748b;">
        Data: S&amp;P 500 + NASDAQ-100 (scraped live from Wikipedia) &nbsp;·&nbsp;
        Model: Random Forest (scikit-learn, {model.n_estimators} trees) &nbsp;·&nbsp;
        Charts: matplotlib
    </div>
    <div style="margin-top:14px;color:#ef4444;font-weight:600;">
        ⚠️ EDUCATIONAL PROJECT ONLY — NOT FINANCIAL ADVICE
    </div>
</div>

</div>
</body>
</html>"""


with open(report_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Report saved: {report_path}")
print(f"   Top pick: {top_picks[0]['ticker'] if top_picks else 'N/A'} (Score: {top_picks[0]['score'] if top_picks else 'N/A'})")

In [ ]:
import os
from IPython.display import HTML, display

display(HTML(html))

full_path = os.path.abspath(report_path)
file_size = os.path.getsize(report_path)

print(f"\n📄 Report file: {full_path}")
print(f"   File size: {file_size / 1024:.1f} KB")
print(f"   Open in browser for best viewing experience")